#### FOURTH ATTEMPT

## Data Modelling

OpTc Dataset
Overview The OpTC (Operationally Transparent Cyber) dataset was developed by Five Directions, under the DARPA Transparent Computing programme, to support research into large-scale cyber-security monitoring and attack detection. It contains endpoint telemetry collected from Windows 10 computers, recording system and network activity through eCAR (extended Cyber Analytics Repository) events. This dataset contains records, including things such as processes, files, network flows, registry activity and other host events. The original release contains roughly a terabyte of compressed data from hundreds of hosts. It includes benign activity as well as red-team attack activity.

I will be using the corrected 2026 version of the OpTC dataset. INRIA reports that the original dataset contains errors involving unique identifiers and other event properties and recommends using the corrected version instead. It is available at: https://entrepot.recherche.data.gouv.fr/dataset.xhtml?persistentId=doi%3A10.57745%2FUXCWOC&utm_source=chatgpt.com

**Project goal:**

The goal of this project is to investigate the use of ML/DL to predict the occurrence of cybersecurity attacks. The project will first analyse and preprocess the OpTC event data to identify behavioural patterns associated with malicious activity, before transforming the sequential telemetry into suitable features and time-based samples for modelling. Different ML/DL approaches will then be evaluated to determine whether patterns in system and network activity can provide sufficient information to identify or predict an impending cyber attack.

**Chapter goal:**

The goal of this chapter is to use the cleaned OpTC telemetry generated in the previous chapter to train and evaluate ML/DL models for malicious event detection. The models will attempt to classify individual telemetry events as benign or malicious. This also allows me to apply the modelling pipeline used for the previous datasets to the more complex OpTC telemetry and compare model performance before moving on to the main attack prediction experiment.

In [1]:
# Imports
%matplotlib inline

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
import random
import tensorflow as tf
import gc

seed = 7
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

# Libraries for splitting, scaling and feature selection
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Libraries for models
from sklearn.naive_bayes import BernoulliNB
from sklearn import tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Libraries for evaluation
from sklearn import metrics

# Libraries for deep learning
from keras.models import Sequential
from keras.layers import Dense, Dropout, BatchNormalization
from keras.layers import Conv1D, MaxPooling1D, Flatten
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from keras.optimizers import Adam

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from pathlib import Path
import pyarrow.parquet as pq


# Ignore warnings
import warnings
warnings.filterwarnings("ignore")





In [2]:
#command used to mount drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Create and save a smaller OpTC sample

In [3]:

# path = Path("/content/drive/MyDrive/solutions")

# # Location of ready flattened parquet files in Google Drive
# data_path = Path("/content/drive/MyDrive/solutions/OpTC_ready")

#  # Specify the location where the sample will be stored
# sample2_path = path / "OpTC_sample2_cleaned"

# # Get all parquet files
# files = list(data_path.glob("*.parquet"))

# sample2_path.mkdir(parents=True, exist_ok=True)


# # Select attack hosts and corresponding control hosts
# selected_hosts = [
#     # Attack hosts
#     # "sysclient0051",
#     # "sysclient0811",

#     "sysclient0201", # September 23
#     "sysclient0501", # September 24
#     "sysclient0351", # September 25 (for testing)
#     "sysclient0051", # September 25 (for training)

#     # Control hosts
#     "sysclient0352",
#     "sysclient0075"
# ]


# # Get only the parquet files belonging to the selected hosts
# selected_files = [
#     file for file in files
#     if any(host in file.name.lower() for host in selected_hosts)
# ]

# print(f"Selected files: {len(selected_files)}")



# # Data Cleaning (Same as third_attempt)
# # =====================================
# missing_counts = {}
# total_rows = 0

# for file in files:

#     parquet_file = pq.ParquetFile(file)
#     metadata = parquet_file.metadata

#     total_rows += metadata.num_rows

#     for i in range(metadata.num_columns):

#         column_name = metadata.schema.column(i).name
#         null_count = 0

#         for row_group in range(metadata.num_row_groups):

#             stats = metadata.row_group(row_group).column(i).statistics

#             if stats is not None and stats.null_count is not None:
#                 null_count += stats.null_count

#         missing_counts[column_name] = (
#             missing_counts.get(column_name, 0) + null_count
#         )
# # Calculate percentage of missing values
# missing_percentage = (
#     pd.Series(missing_counts) / total_rows * 100
# ).sort_values(ascending=False)


# high_missing_cols = missing_percentage[missing_percentage > 99].index.tolist()

# print(f"Columns with >99% missing values: {len(high_missing_cols)}")
# print(high_missing_cols)


# # Specify the columns to drop
# # i.e. columns with many missing and identifier variables
# drop_cols = set(
#     high_missing_cols +
#     ["id", "actorID", "objectID"]
# )

# # ========================================================================

# # Process the selected files one at a time to reduce RAM usage
# for i, file in enumerate(selected_files, start=1):

#     print(f"\n[{i}/{len(selected_files)}] Processing {file.name}")

#     # Load one host at a time
#     data = pd.read_parquet(file)

#     # Drop highly missing and identifier columns
#     data = data.drop(
#         columns=drop_cols,
#         errors="ignore"
#     )

#     # Convert timestamp to datetime
#     data["timestamp"] = pd.to_datetime(
#         data["timestamp"],
#         utc=True
#     )

#     # Arrange each host's events in chronological order
#     data = data.sort_values(
#         "timestamp"
#     ).reset_index(drop=True)

#     # Save each cleaned host separately
#     output_file = sample2_path / file.name

#     data.to_parquet(
#         output_file,
#         index=False
#     )

#     print(
#         f"Saved {file.name} | "
#         f"{len(data):,} rows | "
#         f"{data.shape[1]} columns"
#     )

#     # Clear the current file from memory before loading the next
#     del data


# print("\nDone.")

### Load and preview the OpTC dataset

In [4]:
# Path to the NEW reduced and cleaned OpTC sample2
# Change the folder name here if you saved the new sample under a different name
optc_path = Path(
    "/content/drive/MyDrive/solutions/OpTC_sample2_cleaned"
)

# Find all Parquet files inside the reduced sample folder
files = list(optc_path.glob("*.parquet"))

# Check how many host files were found
print(f"Number of files: {len(files)}")


selected_features = [
    "action",
    "object",
    "acuity_level",
    "direction",
    "info_class",
    "l4protocol",
    "pid",
    "ppid",
    "tid",
    "timestamp",
    "label",
    "hostname"
]

# ============================

# Create a list to temporarily store data from each host/file
data_parts = []

# Load each reduced OpTC file one at a time
for file in files:

    # Read only the features required for the experiment
    part = pd.read_parquet(
        file,
        columns=selected_features
    )

    # Store the current file's DataFrame
    data_parts.append(part)

# Combine all selected host files into one DataFrame
data = pd.concat(
    data_parts,
    ignore_index=True
)

# Release individual parquet dataframes cus of RAM issue
del data_parts

import gc
gc.collect()
print("Dataset shape:", data.shape)

# Check the overall benign/malicious distribution
print("\nLabel distribution:")
print(data["label"].value_counts())

print("\nLabel percentages:")
print(data["label"].value_counts(normalize=True) * 100)


# Identify the source dataset.
# This is metadata and will NOT be used as a model feature
# Rather, it is just to match fields in the other dataset (Windows Optc)
optc = data
optc["dataset_source"] = "OpTC"

del data
gc.collect()

Number of files: 7
Dataset shape: (16555854, 12)

Label distribution:
label
0    16477562
1       78292
Name: count, dtype: int64

Label percentages:
label
0    99.527104
1     0.472896
Name: proportion, dtype: float64


0

### Load and preview the Windows APT dataset


In [5]:
# Load Windows-APT
apt_path = "/content/drive/MyDrive/solutions/Windows_APT/combined.csv"

# Other important files
manifest_path = (
    "/content/drive/MyDrive/solutions/Windows_APT/scenario.csv"
)

validation_path = (
    "/content/drive/MyDrive/solutions/Windows_APT/validation_summary.csv"
)

mapping_path = (
    "/content/drive/MyDrive/solutions/Windows_APT/log_to_scenario_mapping.csv"
)

# Load files

apt = pd.read_csv(
    apt_path,
    low_memory=False
)
manifest = pd.read_csv(manifest_path)
validation = pd.read_csv(validation_path)

mapping = pd.read_csv(
    mapping_path,
    low_memory=False
)

In [6]:
# Combine the scenario description and validation information
scenario_info = manifest.merge(
    validation,
    on=["Scenrario_ID", "Scenario_Name"],
    how="inner"
)



mitre_id_clean = (
    apt["_source.rule.mitre.id"]
    .fillna("")
    .astype(str)
    .str.strip()
)

has_mitre = ~mitre_id_clean.isin(
    ["", "[]", "nan", "None"]
)

# Attach V4 scenario mapping by row position
apt_mapped = pd.concat(
    [
        apt.reset_index(drop=True),

        mapping[
            [
                "Scenario_ID",
                "Scenario_Name",
                "MITRE_Group_ID",
                "Scenario_Posterior",
                "Derived_Label",
                "Attribution_Strength",
                "Mapping_Evidence",
                "Candidate_Scenarios"
            ]
        ].reset_index(drop=True)
    ],
    axis=1
)

# Create proxy binary label for Windows-APT
apt_mapped["label"] = has_mitre.astype(int)

# Mark source dataset
apt_mapped["dataset_source"] = "Windows-APT"

print("Windows-APT proxy label distribution:")
print(apt_mapped["label"].value_counts())

print("\nPercentages:")
print(apt_mapped["label"].value_counts(normalize=True) * 100)



Windows-APT proxy label distribution:
label
1    63619
0    38392
Name: count, dtype: int64

Percentages:
label
1    62.364843
0    37.635157
Name: proportion, dtype: float64


In [7]:
# Rename Windows-APT selected features and metadata
apt_mapped = apt_mapped.rename(columns={
    # Selected features
    "_source.@timestamp": "timestamp",
    "_source.data.win.system.computer": "hostname",
    "_source.rule.description": "rule_description",
    "_source.data.win.system.eventID": "event_id",
    "_source.data.win.eventdata.eventType": "event_type",
    "_source.data.win.eventdata.type": "type",
    "_source.data.win.eventdata.processId": "pid",
    "_source.data.win.eventdata.parentProcessId": "ppid",
    "_source.data.win.system.threadID": "tid",
    "_source.data.win.eventdata.protocol": "protocol",

    # Metadata
    "Scenario_ID": "scenario_id",
    "Scenario_Name": "scenario_name",
    "MITRE_Group_ID": "mitre_group_id",
    "Scenario_Posterior": "scenario_posterior",
    "Derived_Label": "derived_label",
    "Attribution_Strength": "attribution_strength",
    "Mapping_Evidence": "mapping_evidence",
    "Candidate_Scenarios": "candidate_scenarios"
})


# Select relevant Windows-APT features
apt_selected_features = [
    "timestamp",
    "hostname",
    "rule_description",
    "event_id",
    "event_type",
    "type",
    "pid",
    "ppid",
    "tid",
    "protocol",
    "label"
]


# Keep useful metadata for later analysis
metadata_columns = [
    "dataset_source",
    "scenario_id",
    "scenario_name",
    "mitre_group_id",
    "scenario_posterior",
    "derived_label",
    "attribution_strength",
    "mapping_evidence",
    "candidate_scenarios"
]

apt_selected = apt_mapped[apt_selected_features].copy()


# Inspect selected Windows-APT categorical features
for col in [
    "rule_description",
    "event_id",
    "event_type",
    "type",
    "protocol"
]:
    print(f"\n{col}")
    print(
        apt_selected[col]
        .value_counts(dropna=False)
        .head(20)
    )

    # Check missing values in selected Windows-APT features
print(
    apt_selected
    .isnull()
    .sum()
    .sort_values(ascending=False)
)


rule_description
rule_description
Executable file dropped in folder commonly used by malware                                                                                      14714
Summary event of the report's signatures.                                                                                                       14532
Windows command prompt started by an abnormal process                                                                                            6791
Windows Sysmon error event                                                                                                                       5108
Executable dropped in Windows root folder                                                                                                        4659
Windows application error event.                                                                                                                 4190
Registry Value Entry Added to the System                         

### Harmonise OpTC and APT Data

In [8]:
# Create harmonised OpTC dataframe

# Rename OpTC fields to common names
optc.rename(columns={
    "action": "event_action",
    "object": "event_object",
    "l4protocol": "protocol"
}, inplace=True)


# Create behavioural equivalents for Windows-APT
apt_mapped["event_action"] = (
    apt_mapped["rule_description"]
    .fillna(apt_mapped["event_type"])
)

apt_mapped["event_object"] = (
    apt_mapped["event_type"]
    .fillna(apt_mapped["type"])
)


# Features shared across both datasets
harmonised_features = [
    "timestamp",
    "hostname",
    "event_action",
    "event_object",
    "pid",
    "ppid",
    "tid",
    "protocol",
    "label",
    "dataset_source"
]


# Remove unused OpTC columns in place
optc.drop(
    columns=[
        col for col in optc.columns
        if col not in harmonised_features
    ],
    inplace=True
)


# Do not create another copy of the huge OpTC dataframe
optc_harmonised = optc


# Windows-APT is small enough to copy safely
apt_harmonised = apt_mapped[
    harmonised_features
].copy()


gc.collect()


print("OpTC harmonised shape:", optc_harmonised.shape)
print("Windows-APT harmonised shape:", apt_harmonised.shape)

print("\nHarmonised columns:")
print(optc_harmonised.columns.tolist())

print("\nOpTC label distribution:")
print(optc_harmonised["label"].value_counts())

print("\nWindows-APT label distribution:")
print(apt_harmonised["label"].value_counts())

OpTC harmonised shape: (16555854, 10)
Windows-APT harmonised shape: (102011, 10)

Harmonised columns:
['event_action', 'event_object', 'protocol', 'pid', 'ppid', 'tid', 'timestamp', 'label', 'hostname', 'dataset_source']

OpTC label distribution:
label
0    16477562
1       78292
Name: count, dtype: int64

Windows-APT label distribution:
label
1    63619
0    38392
Name: count, dtype: int64


### Create Train-Test Splits

In [9]:
from sklearn.model_selection import GroupShuffleSplit
import gc

# OpTC host-based split
optc_test_hosts = [
    "sysclient0351",
    "sysclient0352"
]

optc_test_mask = optc_harmonised["hostname"].isin(optc_test_hosts)

optc_test = optc_harmonised.loc[optc_test_mask].copy()

# Keep training data in the existing dataframe
optc_harmonised.drop(
    index=optc_harmonised.index[optc_test_mask],
    inplace=True
)

optc_train = optc_harmonised


# Windows-APT scenario-aware split
apt_groups = (
    apt_mapped["scenario_id"]
    .fillna("UNRESOLVED")
    .astype(str)
)

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=7
)

apt_train_idx, apt_test_idx = next(
    splitter.split(
        apt_harmonised,
        apt_harmonised["label"],
        groups=apt_groups
    )
)

apt_train = apt_harmonised.iloc[apt_train_idx].copy()
apt_test = apt_harmonised.iloc[apt_test_idx].copy()


# Release objects no longer needed
del optc_test_mask
gc.collect()


print("OpTC train:", optc_train.shape)
print("OpTC test:", optc_test.shape)

print("\nWindows-APT train:", apt_train.shape)
print("Windows-APT test:", apt_test.shape)

print("\nOpTC train labels:")
print(optc_train["label"].value_counts())

print("\nOpTC test labels:")
print(optc_test["label"].value_counts())

print("\nWindows-APT train labels:")
print(apt_train["label"].value_counts())

print("\nWindows-APT test labels:")
print(apt_test["label"].value_counts())

OpTC train: (16555854, 10)
OpTC test: (0, 10)

Windows-APT train: (65461, 10)
Windows-APT test: (36550, 10)

OpTC train labels:
label
0    16477562
1       78292
Name: count, dtype: int64

OpTC test labels:
Series([], Name: count, dtype: int64)

Windows-APT train labels:
label
1    38334
0    27127
Name: count, dtype: int64

Windows-APT test labels:
label
1    25285
0    11265
Name: count, dtype: int64


### Reduce, combine and save datasets

In [10]:
# Reduce OpTC training data first

optc_malicious = optc_train[
    optc_train["label"] == 1
]

optc_benign = optc_train[
    optc_train["label"] == 0
]

n_benign = min(
    len(optc_benign),
    len(optc_malicious) * 10
)

optc_benign_sample = optc_benign.sample(
    n=n_benign,
    random_state=7
)

optc_train_reduced = pd.concat(
    [
        optc_malicious,
        optc_benign_sample
    ],
    ignore_index=True
)

optc_train_reduced = optc_train_reduced.sample(
    frac=1,
    random_state=7
).reset_index(drop=True)

print("Reduced OpTC train:", optc_train_reduced.shape)
print(optc_train_reduced["label"].value_counts())

Reduced OpTC train: (861212, 10)
label
0    782920
1     78292
Name: count, dtype: int64


In [11]:
# Combine OpTC and Windows-APT training data

combined_train = pd.concat(
    [
        optc_train_reduced,
        apt_train
    ],
    ignore_index=True
)

combined_train = combined_train.sample(
    frac=1,
    random_state=7
).reset_index(drop=True)

print("Combined train shape:", combined_train.shape)

print("\nLabels:")
print(combined_train["label"].value_counts())

print("\nSources:")
print(combined_train["dataset_source"].value_counts())

# Combine untouched OpTC and Windows-APT test data

combined_test = pd.concat(
    [
        optc_test,
        apt_test
    ],
    ignore_index=True
)

print("Combined test shape:", combined_test.shape)

print("\nLabels:")
print(combined_test["label"].value_counts())

print("\nSources:")
print(combined_test["dataset_source"].value_counts())

Combined train shape: (926673, 10)

Labels:
label
0    810047
1    116626
Name: count, dtype: int64

Sources:
dataset_source
OpTC           861212
Windows-APT     65461
Name: count, dtype: int64
Combined test shape: (36550, 10)

Labels:
label
1    25285
0    11265
Name: count, dtype: int64

Sources:
dataset_source
Windows-APT    36550
Name: count, dtype: int64


In [12]:
# Save Combined dataset

# Standardise numeric identifier columns
numeric_cols = [
    "pid",
    "ppid",
    "tid"
]

for col in numeric_cols:

    combined_train[col] = pd.to_numeric(
        combined_train[col],
        errors="coerce"
    ).fillna(-1).astype("int64")

    combined_test[col] = pd.to_numeric(
        combined_test[col],
        errors="coerce"
    ).fillna(-1).astype("int64")


# Standardise timestamp as string
combined_train["timestamp"] = (
    combined_train["timestamp"]
    .astype("string")
    .fillna("")
)

combined_test["timestamp"] = (
    combined_test["timestamp"]
    .astype("string")
    .fillna("")
)


# Standardise categorical columns
categorical_cols = [
    "event_action",
    "event_object",
    "protocol",
    "hostname",
    "dataset_source"
]

for col in categorical_cols:

    combined_train[col] = (
        combined_train[col]
        .astype("string")
        .fillna("")
    )

    combined_test[col] = (
        combined_test[col]
        .astype("string")
        .fillna("")
    )


# Ensure label is integer
combined_train["label"] = (
    pd.to_numeric(
        combined_train["label"],
        errors="coerce"
    )
    .fillna(0)
    .astype("int8")
)

combined_test["label"] = (
    pd.to_numeric(
        combined_test["label"],
        errors="coerce"
    )
    .fillna(0)
    .astype("int8")
)


# Create output folder
combination_path = Path(
    "/content/drive/MyDrive/solutions/Combination_OpTC_APT"
)

combination_path.mkdir(
    parents=True,
    exist_ok=True
)


# Save combined datasets
combined_train.to_parquet(
    combination_path / "combined_train.parquet",
    index=False
)

combined_test.to_parquet(
    combination_path / "combined_test.parquet",
    index=False
)


print("Combined datasets saved successfully.")

print("\nTrain shape:")
print(combined_train.shape)

print("\nTest shape:")
print(combined_test.shape)

print("\nColumn types:")
print(combined_train.dtypes)

gc.collect()

Combined datasets saved successfully.

Train shape:
(926673, 10)

Test shape:
(36550, 10)

Column types:
event_action      string[python]
event_object      string[python]
protocol          string[python]
pid                        int64
ppid                       int64
tid                        int64
timestamp         string[python]
label                       int8
hostname          string[python]
dataset_source    string[python]
dtype: object


0

In [13]:
### Load the combined dataset

In [14]:
combination_path = Path(
    "/content/drive/MyDrive/solutions/Combination_OpTC_APT"
)

train_data = pd.read_parquet(
    combination_path / "combined_train.parquet"
)

test_data = pd.read_parquet(
    combination_path / "combined_test.parquet"
)

print("Train shape:", train_data.shape)
print("Test shape:", test_data.shape)

Train shape: (926673, 10)
Test shape: (36550, 10)


### Prepare the timestamp and feature datatypes

In [15]:
# Convert timestamps

def convert_timestamp(series):

    numeric_timestamp = pd.to_numeric(
        series,
        errors="coerce"
    )

    timestamp = pd.Series(
        pd.NaT,
        index=series.index,
        dtype="datetime64[ns, UTC]"
    )

    numeric_mask = numeric_timestamp.notna()

    if numeric_mask.any():

        values = numeric_timestamp[numeric_mask]

        median_value = values.abs().median()

        if median_value > 1e17:
            unit = "ns"
        elif median_value > 1e14:
            unit = "us"
        elif median_value > 1e11:
            unit = "ms"
        else:
            unit = "s"

        timestamp.loc[numeric_mask] = pd.to_datetime(
            values,
            unit=unit,
            errors="coerce",
            utc=True
        )

    string_mask = ~numeric_mask

    timestamp.loc[string_mask] = pd.to_datetime(
        series[string_mask],
        errors="coerce",
        utc=True
    )

    return timestamp


train_data["timestamp"] = convert_timestamp(
    train_data["timestamp"]
)

test_data["timestamp"] = convert_timestamp(
    test_data["timestamp"]
)

In [16]:
# Extract time features

for data in [train_data, test_data]:

    data["hour"] = (
        data["timestamp"]
        .dt.hour
        .fillna(-1)
        .astype("float32")
    )

    data["minute"] = (
        data["timestamp"]
        .dt.minute
        .fillna(-1)
        .astype("float32")
    )

# Ensure numeric fields are numeric

for col in ["pid", "ppid", "tid"]:

    train_data[col] = pd.to_numeric(
        train_data[col],
        errors="coerce"
    ).fillna(-1).astype("float32")

    test_data[col] = pd.to_numeric(
        test_data[col],
        errors="coerce"
    ).fillna(-1).astype("float32")


# Standardise categorical fields

for col in [
    "event_action",
    "event_object",
    "protocol"
]:

    train_data[col] = (
        train_data[col]
        .fillna("UNKNOWN")
        .astype(str)
    )

    test_data[col] = (
        test_data[col]
        .fillna("UNKNOWN")
        .astype(str)
    )

### Create X and Y

In [17]:
# Extract hour and minute from timestamp
for dataset in [train_data, test_data]:
    dataset["hour"] = dataset["timestamp"].dt.hour.astype("float32")
    dataset["minute"] = dataset["timestamp"].dt.minute.astype("float32")


# Remove target, raw time and identifier variables
drop_columns = [
    "label",
    "timestamp",
    "hostname",
    "dataset_source"
]


x_train = train_data.drop(
    columns=drop_columns,
    errors="ignore"
)

Y = train_data["label"].astype("int8")


x_test = test_data.drop(
    columns=drop_columns,
    errors="ignore"
)

Y_TEST = test_data["label"].astype("int8")


print("Training shape:", x_train.shape)
print("Test shape:", x_test.shape)
print("Features:", x_train.columns.tolist())

# Remove time and identifier variables
drop_columns = [
    "label",
    "timestamp",
    "hostname",
    "dataset_source"
]


x_train = train_data.drop(
    columns=drop_columns,
    errors="ignore"
)

Y = train_data["label"].astype("int8")


x_test = test_data.drop(
    columns=drop_columns,
    errors="ignore"
)

Y_TEST = test_data["label"].astype("int8")


print("Training shape:", x_train.shape)
print("Test shape:", x_test.shape)
print("Features:", x_train.columns.tolist())

Training shape: (926673, 8)
Test shape: (36550, 8)
Features: ['event_action', 'event_object', 'protocol', 'pid', 'ppid', 'tid', 'hour', 'minute']
Training shape: (926673, 8)
Test shape: (36550, 8)
Features: ['event_action', 'event_object', 'protocol', 'pid', 'ppid', 'tid', 'hour', 'minute']


### Encode Categorical Features

In [18]:


# Identify categorical and numerical columns
cat_cols = x_train.select_dtypes(
    include=["object", "string"]
).columns

numeric_cols = x_train.select_dtypes(
    include=np.number
).columns


# Numerical preprocessing
numeric_transformer = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    )
])


# Categorical preprocessing
categorical_transformer = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="constant",
            fill_value="UNKNOWN"
        )
    ),
    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore",
            dtype=np.float32
        )
    )
])


# Combine preprocessing
preprocessor = ColumnTransformer([
    (
        "cat",
        categorical_transformer,
        cat_cols
    ),
    (
        "num",
        numeric_transformer,
        numeric_cols
    )
], sparse_threshold=1.0)


# Fit ONLY on training data
X = preprocessor.fit_transform(
    x_train
).astype(np.float32)

# Apply same preprocessing to test data
X_TEST = preprocessor.transform(
    x_test
).astype(np.float32)


print(X.shape)
print(X_TEST.shape)

(926673, 1474)
(36550, 1474)


### Define and Train ML models

In [19]:
# Define models
models = [
    ("Logistic Regression", LogisticRegression(
        max_iter=1000,
        random_state=7,
        class_weight="balanced"
    )),

    ("Bernoulli NB", BernoulliNB()),

    ("Decision Tree", DecisionTreeClassifier(
        random_state=7,
        class_weight="balanced",
        max_depth=10
    )),

    ("Random Forest", RandomForestClassifier(
        n_estimators=100,
        random_state=7,
        class_weight="balanced",
        n_jobs=-1
    )),

    ("XGBoost", XGBClassifier(
        random_state=7,
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        eval_metric="logloss",
        n_jobs=-1
    ))
]

In [20]:
# Train and evaluate the models on training data

for name, model in models:

    # Train model
    model.fit(X, Y)

    # Make predictions on training data
    predictions = model.predict(X)

    accuracy = metrics.accuracy_score(Y, predictions)
    conf_matrix = metrics.confusion_matrix(Y, predictions)
    report = metrics.classification_report(Y, predictions)

    print(f"\n===== {name} - Train Evaluation =====")
    print("Accuracy:", accuracy)
    print("Confusion Matrix:\n", conf_matrix)
    print("Classification Report:\n", report)


===== Logistic Regression - Train Evaluation =====
Accuracy: 0.9147725249359806
Confusion Matrix:
 [[735607  74440]
 [  4538 112088]]
Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.91      0.95    810047
           1       0.60      0.96      0.74    116626

    accuracy                           0.91    926673
   macro avg       0.80      0.93      0.84    926673
weighted avg       0.94      0.91      0.92    926673


===== Bernoulli NB - Train Evaluation =====
Accuracy: 0.8974352333563188
Confusion Matrix:
 [[726091  83956]
 [ 11088 105538]]
Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.90      0.94    810047
           1       0.56      0.90      0.69    116626

    accuracy                           0.90    926673
   macro avg       0.77      0.90      0.81    926673
weighted avg       0.93      0.90      0.91    926673


===== Decision Tree - Train Evalua

### Validate on Test data

In [21]:
# Evaluate the models on test data

for name, model in models:

    predictions = model.predict(X_TEST)

    accuracy = metrics.accuracy_score(Y_TEST, predictions)
    conf_matrix = metrics.confusion_matrix(Y_TEST, predictions)
    report = metrics.classification_report(Y_TEST, predictions)

    print(f"\n===== {name} - Test Evaluation =====")
    print("Accuracy:", accuracy)
    print("Confusion Matrix:\n", conf_matrix)
    print("Classification Report:\n", report)


===== Logistic Regression - Test Evaluation =====
Accuracy: 0.9893296853625171
Confusion Matrix:
 [[11096   169]
 [  221 25064]]
Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.98      0.98     11265
           1       0.99      0.99      0.99     25285

    accuracy                           0.99     36550
   macro avg       0.99      0.99      0.99     36550
weighted avg       0.99      0.99      0.99     36550


===== Bernoulli NB - Test Evaluation =====
Accuracy: 0.9630916552667579
Confusion Matrix:
 [[ 9917  1348]
 [    1 25284]]
Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.88      0.94     11265
           1       0.95      1.00      0.97     25285

    accuracy                           0.96     36550
   macro avg       0.97      0.94      0.96     36550
weighted avg       0.96      0.96      0.96     36550


===== Decision Tree - Test Evaluation =====


### Reduce data for DL models

In [22]:
# Reduce the data for ANN/CNN because the full dataset exceeds RAM
np.random.seed(7)

n_train = min(200000, X.shape[0])
n_test = min(200000, X_TEST.shape[0])

train_idx = np.random.choice(
    X.shape[0],
    n_train,
    replace=False
)

test_idx = np.random.choice(
    X_TEST.shape[0],
    n_test,
    replace=False
)

X = X[train_idx]
X_TEST = X_TEST[test_idx]

# Convert to dense only if sparse
if hasattr(X, "toarray"):
    X = X.toarray()

if hasattr(X_TEST, "toarray"):
    X_TEST = X_TEST.toarray()

X = X.astype("float32")
X_TEST = X_TEST.astype("float32")

Y = Y.iloc[train_idx].reset_index(drop=True)
Y_TEST = Y_TEST.iloc[test_idx].reset_index(drop=True)

print("DL training shape:", X.shape)
print("DL test shape:", X_TEST.shape)

DL training shape: (200000, 1474)
DL test shape: (36550, 1474)


### DL Models - ANN

In [23]:
# Build ANN
model = Sequential()

model.add(Dense(128, activation='relu', input_shape=(X.shape[1],)))
model.add(BatchNormalization())
model.add(Dropout(0.3))

model.add(Dense(64, activation='relu'))
model.add(Dropout(0.3))

model.add(Dense(1, activation='sigmoid'))

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Train ANN
history = model.fit(
    X, Y,
    epochs=10,
    batch_size=64,
    validation_split=0.2
)

# Train evaluation
train_pred = (model.predict(X) > 0.5).astype(int)

print("Train Accuracy:", metrics.accuracy_score(Y, train_pred))
print(metrics.classification_report(Y, train_pred))

# Test evaluation
test_pred = (model.predict(X_TEST) > 0.5).astype(int)

print("Test Accuracy:", metrics.accuracy_score(Y_TEST, test_pred))
print(metrics.classification_report(Y_TEST, test_pred))

Epoch 1/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 14s 5ms/step - accuracy: 0.9746 - loss: 0.0627 - val_accuracy: 0.9836 - val_loss: 0.0381
Epoch 2/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - accuracy: 0.9840 - loss: 0.0405 - val_accuracy: 0.9857 - val_loss: 0.0329
Epoch 3/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - accuracy: 0.9856 - loss: 0.0361 - val_accuracy: 0.9888 - val_loss: 0.0279
Epoch 4/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - accuracy: 0.9872 - loss: 0.0330 - val_accuracy: 0.9892 - val_loss: 0.0277
Epoch 5/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - accuracy: 0.9883 - loss: 0.0309 - val_accuracy: 0.9907 - val_loss: 0.0277
Epoch 6/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 20s 4ms/step - accuracy: 0.9887 - loss: 0.0293 - val_accuracy: 0.9913 - val_loss: 0.0247
Epoch 7/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - accuracy: 0.9894 - loss: 0.0279 - val_accuracy: 0.9916 - val_loss: 0.0240
Epoch 8/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - accuracy: 0.9899 - loss: 0

### 12. DL Models - CNN

In [24]:
# Reshape input for Conv1D
X_cnn = np.expand_dims(X, axis=2)
X_TEST_cnn = np.expand_dims(X_TEST, axis=2)


# Build the CNN
cnn_classifier = Sequential()

cnn_classifier.add(
    Conv1D(
        32,
        kernel_size=3,
        activation='relu',
        padding='same',
        input_shape=(X_cnn.shape[1], 1)
    )
)

cnn_classifier.add(BatchNormalization())
cnn_classifier.add(MaxPooling1D(pool_size=2))
cnn_classifier.add(Dropout(0.2))

cnn_classifier.add(
    Conv1D(
        64,
        kernel_size=3,
        activation='relu',
        padding='same'
    )
)

cnn_classifier.add(BatchNormalization())
cnn_classifier.add(MaxPooling1D(pool_size=2))
cnn_classifier.add(Dropout(0.2))

cnn_classifier.add(Flatten())
cnn_classifier.add(Dense(64, activation='relu'))
cnn_classifier.add(Dropout(0.3))

cnn_classifier.add(Dense(1, activation='sigmoid'))


# Compile the model
cnn_classifier.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)


# Train the model
cnn_history = cnn_classifier.fit(
    X_cnn,
    Y,
    epochs=10,
    batch_size=64,
    validation_split=0.2
)


# Train evaluation
train_pred = (cnn_classifier.predict(X_cnn) > 0.5).astype(int)

print("Train Accuracy:", metrics.accuracy_score(Y, train_pred))
print(metrics.classification_report(Y, train_pred))


# Test evaluation
test_pred = (cnn_classifier.predict(X_TEST_cnn) > 0.5).astype(int)

print("Test Accuracy:", metrics.accuracy_score(Y_TEST, test_pred))
print(metrics.classification_report(Y_TEST, test_pred))

Epoch 1/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 371s 148ms/step - accuracy: 0.9630 - loss: 0.0899 - val_accuracy: 0.8202 - val_loss: 3.4430
Epoch 2/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 368s 142ms/step - accuracy: 0.9797 - loss: 0.0548 - val_accuracy: 0.1259 - val_loss: 86.4959
Epoch 3/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 383s 143ms/step - accuracy: 0.9836 - loss: 0.0450 - val_accuracy: 0.9851 - val_loss: 0.0377
Epoch 4/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 382s 143ms/step - accuracy: 0.9853 - loss: 0.0396 - val_accuracy: 0.9851 - val_loss: 0.0574
Epoch 5/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 358s 143ms/step - accuracy: 0.9873 - loss: 0.0359 - val_accuracy: 0.9904 - val_loss: 0.0274
Epoch 6/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 373s 140ms/step - accuracy: 0.9877 - loss: 0.0339 - val_accuracy: 0.9293 - val_loss: 0.4974
Epoch 7/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 386s 141ms/step - accuracy: 0.9891 - loss: 0.0310 - val_accuracy: 0.9919 - val_loss: 0.0237
Epoch 8/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 377s 139ms/step - a

### 13. Results Presentation

In [25]:
# Generate summary tables for presentation

train_results = []
test_results = []

# Save ANN model before looping through classical models
ann_classifier = model


# Classical ML models
for name, classifier in models:

    train_predictions = classifier.predict(X)
    test_predictions = classifier.predict(X_TEST)

    train_results.append([
        name,
        metrics.accuracy_score(Y, train_predictions),
        metrics.precision_score(Y, train_predictions, average="macro", zero_division=0),
        metrics.recall_score(Y, train_predictions, average="macro", zero_division=0),
        metrics.f1_score(Y, train_predictions, average="macro", zero_division=0)
    ])

    test_results.append([
        name,
        metrics.accuracy_score(Y_TEST, test_predictions),
        metrics.precision_score(Y_TEST, test_predictions, average="macro", zero_division=0),
        metrics.recall_score(Y_TEST, test_predictions, average="macro", zero_division=0),
        metrics.f1_score(Y_TEST, test_predictions, average="macro", zero_division=0)
    ])


# ANN predictions
ann_train_pred = (
    ann_classifier.predict(X, verbose=0) > 0.5
).astype(int).ravel()

ann_test_pred = (
    ann_classifier.predict(X_TEST, verbose=0) > 0.5
).astype(int).ravel()

train_results.append([
    "ANN",
    metrics.accuracy_score(Y, ann_train_pred),
    metrics.precision_score(Y, ann_train_pred, average="macro", zero_division=0),
    metrics.recall_score(Y, ann_train_pred, average="macro", zero_division=0),
    metrics.f1_score(Y, ann_train_pred, average="macro", zero_division=0)
])

test_results.append([
    "ANN",
    metrics.accuracy_score(Y_TEST, ann_test_pred),
    metrics.precision_score(Y_TEST, ann_test_pred, average="macro", zero_division=0),
    metrics.recall_score(Y_TEST, ann_test_pred, average="macro", zero_division=0),
    metrics.f1_score(Y_TEST, ann_test_pred, average="macro", zero_division=0)
])


# CNN predictions
cnn_train_pred = (
    cnn_classifier.predict(X_cnn, verbose=0) > 0.5
).astype(int).ravel()

cnn_test_pred = (
    cnn_classifier.predict(X_TEST_cnn, verbose=0) > 0.5
).astype(int).ravel()

train_results.append([
    "CNN",
    metrics.accuracy_score(Y, cnn_train_pred),
    metrics.precision_score(Y, cnn_train_pred, average="macro", zero_division=0),
    metrics.recall_score(Y, cnn_train_pred, average="macro", zero_division=0),
    metrics.f1_score(Y, cnn_train_pred, average="macro", zero_division=0)
])

test_results.append([
    "CNN",
    metrics.accuracy_score(Y_TEST, cnn_test_pred),
    metrics.precision_score(Y_TEST, cnn_test_pred, average="macro", zero_division=0),
    metrics.recall_score(Y_TEST, cnn_test_pred, average="macro", zero_division=0),
    metrics.f1_score(Y_TEST, cnn_test_pred, average="macro", zero_division=0)
])


# Create presentation tables
columns = [
    "Model",
    "Accuracy",
    "Macro Precision",
    "Macro Recall",
    "Macro F1"
]

train_results = pd.DataFrame(
    train_results,
    columns=columns
).round(4)

test_results = pd.DataFrame(
    test_results,
    columns=columns
).round(4)


print("TRAIN RESULTS")
display(train_results)

print("TEST RESULTS")
display(test_results)

TRAIN RESULTS


,Model,Accuracy,Macro Precision,Macro Recall,Macro F1
0,Logistic Regression,0.9140,0.7962,0.9331,0.8429
1,Bernoulli NB,0.8960,0.7690,0.8986,0.8119
2,Decision Tree,0.9724,0.9110,0.9823,0.9424
3,Random Forest,1.0000,0.9999,1.0000,1.0000
4,XGBoost,0.1259,0.0630,0.5000,0.1118
5,ANN,0.9931,0.9801,0.9889,0.9845
6,CNN,0.9921,0.9835,0.9806,0.9820


TEST RESULTS


,Model,Accuracy,Macro Precision,Macro Recall,Macro F1
0,Logistic Regression,0.9893,0.9869,0.9881,0.9875
1,Bernoulli NB,0.9631,0.9746,0.9401,0.9552
2,Decision Tree,0.9436,0.9603,0.9098,0.9303
3,Random Forest,0.9550,0.9364,0.9674,0.9492
4,XGBoost,0.6918,0.3459,0.5000,0.4089
5,ANN,0.9485,0.9284,0.9626,0.9421
6,CNN,0.9367,0.9149,0.9531,0.9293
